# Problem Statement

## **Business Context**

"Visit with Us," is launching a Wellness Tourism Package and wants to predict which customers are likely to purchase it before the sales team contacts them. An MLOps workflow is needed so that dataset registration, data preparation, model training, experiment logging, model registration, deployment, and CI/CD can run in a repeatable and scalable way.

## **Objective**

The goal of this project is to build an end-to-end MLOps pipeline using GitHub, Hugging Face Hub/Spaces, and a machine learning model that predicts `ProdTaken` (0 or 1). The notebook below covers data registration, data preparation, model experimentation, deployment assets, and the GitHub Actions workflow required for automation.

## **Data Description**

The dataset contains customer demographics and interaction history. The target variable is `ProdTaken`, which indicates whether the customer purchased the package. The project uses the algorithms allowed in the rubric: Random Forest, Gradient Boosting, and AdaBoost.

# Model Building

In [ ]:
# Create the required project folders
from pathlib import Path

PROJECT_ROOT = Path("tourism_project")
for folder in [
    PROJECT_ROOT,
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "model_building",
    PROJECT_ROOT / "deployment",
    PROJECT_ROOT / ".github" / "workflows",
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Created project structure under:", PROJECT_ROOT.resolve())

In [ ]:
# Project configuration
HF_USERNAME = "YOUR_HF_USERNAME"
HF_DATASET_REPO_ID = f"{HF_USERNAME}/tourism-package-data"
HF_MODEL_REPO_ID = f"{HF_USERNAME}/tourism-package-model"
HF_SPACE_REPO_ID = f"{HF_USERNAME}/tourism-package-app"
GITHUB_REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME"
HF_TOKEN = None  # replace with your token only when you are ready to upload

print("Dataset repo:", HF_DATASET_REPO_ID)
print("Model repo:", HF_MODEL_REPO_ID)
print("Space repo:", HF_SPACE_REPO_ID)

## Data Registration

In [ ]:
# Copy the source dataset into the required data folder
import shutil
from pathlib import Path

source_candidates = [Path("tourism.csv"), Path("/mnt/data/tourism.csv")]
source_file = next((p for p in source_candidates if p.exists()), None)
if source_file is None:
    raise FileNotFoundError("tourism.csv was not found. Upload it and rerun this cell.")

local_dataset_path = PROJECT_ROOT / "data" / "tourism.csv"
shutil.copy2(source_file, local_dataset_path)
print("Dataset copied to:", local_dataset_path)

In [ ]:
# Optional: register the dataset on Hugging Face Hub
from huggingface_hub import HfApi

if HF_TOKEN and "YOUR_HF_USERNAME" not in HF_DATASET_REPO_ID:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_DATASET_REPO_ID, repo_type="dataset", exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(local_dataset_path),
        path_in_repo="tourism.csv",
        repo_id=HF_DATASET_REPO_ID,
        repo_type="dataset",
    )
    print(f"Dataset uploaded to https://huggingface.co/datasets/{HF_DATASET_REPO_ID}")
else:
    print("Skipped Hugging Face upload. Add a valid HF token and repo id to enable registration.")

## Data Preparation

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load directly from HF when available; otherwise fall back to the local data file
try:
    from huggingface_hub import hf_hub_download
    if HF_TOKEN and "YOUR_HF_USERNAME" not in HF_DATASET_REPO_ID:
        data_path = hf_hub_download(repo_id=HF_DATASET_REPO_ID, filename="tourism.csv", repo_type="dataset", token=HF_TOKEN)
    else:
        data_path = str(local_dataset_path)
except Exception:
    data_path = str(local_dataset_path)

raw_df = pd.read_csv(data_path)
print("Raw shape:", raw_df.shape)
raw_df.head()

In [ ]:
# Clean the data and remove unnecessary columns
clean_df = raw_df.copy()
columns_to_drop = [c for c in ["Unnamed: 0", "CustomerID"] if c in clean_df.columns]
clean_df = clean_df.drop(columns=columns_to_drop)

print("Dropped columns:", columns_to_drop)
print("Cleaned shape:", clean_df.shape)
print("Missing values by column:")
print(clean_df.isna().sum())

In [ ]:
# Split into train and test sets and save locally
train_df, test_df = train_test_split(
    clean_df,
    test_size=0.20,
    random_state=42,
    stratify=clean_df["ProdTaken"]
)

train_path = PROJECT_ROOT / "data" / "train.csv"
test_path = PROJECT_ROOT / "data" / "test.csv"
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train saved at:", train_path)
print("Test saved at:", test_path)

In [ ]:
# Optional: upload train and test datasets back to Hugging Face Hub
from huggingface_hub import HfApi

if HF_TOKEN and "YOUR_HF_USERNAME" not in HF_DATASET_REPO_ID:
    api = HfApi(token=HF_TOKEN)
    for file_name, file_path in {"train.csv": train_path, "test.csv": test_path}.items():
        api.upload_file(
            path_or_fileobj=str(file_path),
            path_in_repo=file_name,
            repo_id=HF_DATASET_REPO_ID,
            repo_type="dataset",
        )
    print(f"Train and test files uploaded to https://huggingface.co/datasets/{HF_DATASET_REPO_ID}")
else:
    print("Skipped train/test upload. Add a valid HF token and repo id to enable upload.")

**Observation:** The raw dataset has 4,128 rows and 21 columns. `Unnamed: 0` and `CustomerID` are not useful for modeling, so they are removed. The final train-test split preserves the class distribution using stratification.

## Model Training and Registration with Experimentation Tracking

In [ ]:
# Load the train and test datasets from Hugging Face if available, otherwise use local copies
try:
    from huggingface_hub import hf_hub_download
    if HF_TOKEN and "YOUR_HF_USERNAME" not in HF_DATASET_REPO_ID:
        train_data_path = hf_hub_download(repo_id=HF_DATASET_REPO_ID, filename="train.csv", repo_type="dataset", token=HF_TOKEN)
        test_data_path = hf_hub_download(repo_id=HF_DATASET_REPO_ID, filename="test.csv", repo_type="dataset", token=HF_TOKEN)
    else:
        train_data_path, test_data_path = str(train_path), str(test_path)
except Exception:
    train_data_path, test_data_path = str(train_path), str(test_path)

train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)

X_train = train_df.drop(columns=["ProdTaken"])
y_train = train_df["ProdTaken"]
X_test = test_df.drop(columns=["ProdTaken"])
y_test = test_df["ProdTaken"]

print(X_train.shape, X_test.shape)

In [ ]:
# Build preprocessing pipelines
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ]
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
# Define candidate models and tuning grids
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

model_grid = {
    "RandomForest": (
        RandomForestClassifier(random_state=42, class_weight="balanced"),
        {
            "model__n_estimators": [200, 300],
            "model__max_depth": [10, 20],
            "model__min_samples_split": [2, 5],
            "model__min_samples_leaf": [1, 2],
        },
    ),
    "GradientBoosting": (
        GradientBoostingClassifier(random_state=42),
        {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.05, 0.1],
            "model__max_depth": [3],
        },
    ),
    "AdaBoost": (
        AdaBoostClassifier(random_state=42),
        {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.05, 0.1, 0.5],
        },
    ),
}

In [ ]:
# Tune the models, log the parameters, and evaluate them
import json
import joblib
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
    mlflow.set_experiment("tourism_package_prediction")
except Exception:
    MLFLOW_AVAILABLE = False
    print("mlflow is not installed in this runtime. Results will be saved to a CSV file instead.")

results = []
best_model = None
best_model_name = None
best_f1 = -1

for model_name, (model_obj, param_grid) in model_grid.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model_obj)
    ])

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=3,
        scoring="f1",
        n_jobs=1
    )

    if MLFLOW_AVAILABLE:
        with mlflow.start_run(run_name=model_name):
            grid.fit(X_train, y_train)
            preds = grid.predict(X_test)
            probs = grid.predict_proba(X_test)[:, 1]
            metrics = {
                "accuracy": accuracy_score(y_test, preds),
                "precision": precision_score(y_test, preds),
                "recall": recall_score(y_test, preds),
                "f1": f1_score(y_test, preds),
                "roc_auc": roc_auc_score(y_test, probs),
            }
            mlflow.log_params(grid.best_params_)
            mlflow.log_metrics(metrics)
            mlflow.sklearn.log_model(grid.best_estimator_, artifact_path=f"{model_name}_model")
    else:
        grid.fit(X_train, y_train)
        preds = grid.predict(X_test)
        probs = grid.predict_proba(X_test)[:, 1]
        metrics = {
            "accuracy": accuracy_score(y_test, preds),
            "precision": precision_score(y_test, preds),
            "recall": recall_score(y_test, preds),
            "f1": f1_score(y_test, preds),
            "roc_auc": roc_auc_score(y_test, probs),
        }

    row = {"model": model_name, **grid.best_params_, **metrics}
    results.append(row)

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_model_name = model_name
        best_model = grid.best_estimator_

results_df = pd.DataFrame(results).sort_values("f1", ascending=False).reset_index(drop=True)
results_df

In [ ]:
# Save the best model and the experimentation results
model_output_path = PROJECT_ROOT / "model_building" / "best_model.joblib"
results_output_path = PROJECT_ROOT / "model_building" / "experiment_results.csv"
info_output_path = PROJECT_ROOT / "model_building" / "best_model_info.json"

joblib.dump(best_model, model_output_path)
results_df.to_csv(results_output_path, index=False)

with open(info_output_path, "w") as f:
    json.dump({"best_model_name": best_model_name, "best_f1": float(best_f1)}, f, indent=2)

print("Best model:", best_model_name)
print("Best F1-score:", round(best_f1, 4))
print("Saved model to:", model_output_path)

In [ ]:
# Optional: register the best model in the Hugging Face model hub
from huggingface_hub import HfApi

if HF_TOKEN and "YOUR_HF_USERNAME" not in HF_MODEL_REPO_ID:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_MODEL_REPO_ID, repo_type="model", exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(model_output_path),
        path_in_repo="best_model.joblib",
        repo_id=HF_MODEL_REPO_ID,
        repo_type="model",
    )
    api.upload_file(
        path_or_fileobj=str(results_output_path),
        path_in_repo="experiment_results.csv",
        repo_id=HF_MODEL_REPO_ID,
        repo_type="model",
    )
    print(f"Model uploaded to https://huggingface.co/{HF_MODEL_REPO_ID}")
else:
    print("Skipped model upload. Add a valid HF token and repo id to enable model registration.")

**Observation:** Among the tested models, Random Forest gave the strongest performance on the held-out test set with the highest F1-score and ROC-AUC, so it was selected as the final model for deployment.

# Deployment

## Dockerfile

In [ ]:
dockerfile_text = """
FROM python:3.10-slim

WORKDIR /app
COPY . /app
RUN pip install --no-cache-dir -r requirements.txt
EXPOSE 8501
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]
""".strip()

(PROJECT_ROOT / "deployment" / "Dockerfile").write_text(dockerfile_text + "
")
print((PROJECT_ROOT / "deployment" / "Dockerfile").read_text())

## Streamlit App

In [ ]:
app_code = """
import os
import joblib
import pandas as pd
import streamlit as st
from huggingface_hub import hf_hub_download

st.set_page_config(page_title="Tourism Package Prediction", layout="centered")
st.title("Wellness Tourism Package Prediction")
st.write("Predict whether a customer is likely to purchase the package.")

@st.cache_resource
def load_model():
    repo_id = os.getenv("HF_MODEL_REPO_ID", "YOUR_HF_USERNAME/tourism-package-model")
    token = os.getenv("HF_TOKEN")
    try:
        model_path = hf_hub_download(repo_id=repo_id, filename="best_model.joblib", token=token)
    except Exception:
        model_path = "best_model.joblib"
    return joblib.load(model_path)

model = load_model()

input_data = {
    "Age": st.number_input("Age", min_value=18, max_value=80, value=32),
    "TypeofContact": st.selectbox("Type of Contact", ["Company Invited", "Self Enquiry", "Self Inquiry"]),
    "CityTier": st.selectbox("City Tier", [1, 2, 3]),
    "DurationOfPitch": st.number_input("Duration of Pitch", min_value=5, max_value=300, value=20),
    "Occupation": st.selectbox("Occupation", ["Salaried", "Small Business", "Large Business", "Free Lancer"]),
    "Gender": st.selectbox("Gender", ["Male", "Female"]),
    "NumberOfPersonVisiting": st.number_input("Number of Persons Visiting", min_value=1, max_value=10, value=2),
    "NumberOfFollowups": st.number_input("Number of Followups", min_value=0, max_value=10, value=3),
    "ProductPitched": st.selectbox("Product Pitched", ["Basic", "Standard", "Deluxe", "Super Deluxe", "King"]),
    "PreferredPropertyStar": st.selectbox("Preferred Property Star", [3, 4, 5]),
    "MaritalStatus": st.selectbox("Marital Status", ["Single", "Married", "Divorced", "Unmarried"]),
    "NumberOfTrips": st.number_input("Number of Trips", min_value=0, max_value=20, value=2),
    "Passport": st.selectbox("Passport", [0, 1]),
    "PitchSatisfactionScore": st.selectbox("Pitch Satisfaction Score", [1, 2, 3, 4, 5]),
    "OwnCar": st.selectbox("Own Car", [0, 1]),
    "NumberOfChildrenVisiting": st.number_input("Number of Children Visiting", min_value=0, max_value=5, value=0),
    "Designation": st.selectbox("Designation", ["Executive", "Manager", "Senior Manager", "AVP", "VP"]),
    "MonthlyIncome": st.number_input("Monthly Income", min_value=1000, max_value=500000, value=25000),
}

input_df = pd.DataFrame([input_data])
st.subheader("Input Data")
st.dataframe(input_df)

if st.button("Predict"):
    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0, 1]
    st.metric("Probability of Purchase", f"{probability:.2%}")
    if prediction == 1:
        st.success("This customer is likely to purchase the Wellness Tourism Package.")
    else:
        st.warning("This customer is unlikely to purchase the Wellness Tourism Package.")
""".strip()

(PROJECT_ROOT / "deployment" / "app.py").write_text(app_code + "
")
print("app.py created successfully")

## Dependency Handling

In [ ]:
requirements_text = """
pandas
numpy
scikit-learn
joblib
huggingface_hub
streamlit
mlflow
""".strip()

(PROJECT_ROOT / "deployment" / "requirements.txt").write_text(requirements_text + "
")
(PROJECT_ROOT / "requirements.txt").write_text(requirements_text + "
")
print((PROJECT_ROOT / "deployment" / "requirements.txt").read_text())

# Hosting

In [ ]:
push_script = """
import os
from huggingface_hub import HfApi

SPACE_REPO_ID = os.getenv("HF_SPACE_REPO_ID", "YOUR_HF_USERNAME/tourism-package-app")
HF_TOKEN = os.getenv("HF_TOKEN")

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=SPACE_REPO_ID, repo_type="space", space_sdk="streamlit", exist_ok=True)
api.upload_folder(folder_path="tourism_project/deployment", repo_id=SPACE_REPO_ID, repo_type="space")
print(f"Deployment files pushed to https://huggingface.co/spaces/{SPACE_REPO_ID}")
""".strip()

(PROJECT_ROOT / "deployment" / "push_to_space.py").write_text(push_script + "
")
print("push_to_space.py created successfully")

# MLOps Pipeline with Github Actions Workflow

In [ ]:
pipeline_text = """
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main

jobs:
  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Upload dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_DATASET_REPO_ID: ${{ secrets.HF_DATASET_REPO_ID }}
        run: python scripts_register_data.py

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Run data preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_DATASET_REPO_ID: ${{ secrets.HF_DATASET_REPO_ID }}
        run: python scripts_data_prep.py

  model-training:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Train model and upload best artifact
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_DATASET_REPO_ID: ${{ secrets.HF_DATASET_REPO_ID }}
          HF_MODEL_REPO_ID: ${{ secrets.HF_MODEL_REPO_ID }}
        run: python scripts_train.py

  deploy-space:
    needs: model-training
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Push deployment files to Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_SPACE_REPO_ID: ${{ secrets.HF_SPACE_REPO_ID }}
        run: python tourism_project/deployment/push_to_space.py
""".strip()

workflow_path = PROJECT_ROOT / ".github" / "workflows" / "pipeline.yml"
workflow_path.parent.mkdir(parents=True, exist_ok=True)
workflow_path.write_text(pipeline_text + "
")
print(workflow_path.read_text())

## Requirements file for the Github Actions Workflow

In [ ]:
# Reuse the root requirements.txt for GitHub Actions
print((PROJECT_ROOT / "requirements.txt").read_text())

## Github Authentication and Push Files

In [ ]:
github_instructions = """
1. Create a GitHub repository.
2. Add these secrets in GitHub: HF_TOKEN, HF_DATASET_REPO_ID, HF_MODEL_REPO_ID, HF_SPACE_REPO_ID.
3. Copy the contents of tourism_project into your repository.
4. Commit and push to the main branch.
5. GitHub Actions will automatically run the pipeline defined in .github/workflows/pipeline.yml.
""".strip()

(PROJECT_ROOT / "github_push_instructions.txt").write_text(github_instructions + "
")
print(github_instructions)

# Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)

In [ ]:
print("Add your GitHub repository link here:")
print(GITHUB_REPO_URL)

- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

In [ ]:
print("Add your Hugging Face Space link here after deployment:")
print(f"https://huggingface.co/spaces/{HF_SPACE_REPO_ID}")

## Final Insights

1. The dataset is moderately imbalanced, so stratified splitting is important.
2. Removing identifier-style columns avoids leakage and noise.
3. Tree-based ensemble models are a strong fit because the data contains both numeric and categorical features after preprocessing.
4. Random Forest delivered the best balance of recall and precision in local testing, making it a good deployment choice for this business problem.
5. The full project is production-oriented because it includes Hub registration, experiment tracking, model packaging, Streamlit deployment, and CI/CD automation.

<font size=6 color="navyblue">Power Ahead!</font>
___